# QMIX — Smart Warehouse
Monotonic Value Function Factorisation for Multi-Agent RL.

**Run on Colab GPU for best performance.**

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────
import os
if not os.path.exists('/content/smart-warehouse'):
    !git clone https://github.com/yjkim717/smart-warehouse.git
else:
    !cd /content/smart-warehouse && git pull origin main

!pip install rware gymnasium pyyaml -q

import sys
sys.path.insert(0, '/content/smart-warehouse')
print('Setup done.')

In [ ]:
# ── Imports & Config ──────────────────────────────────────────────
import yaml, json, time, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import random
import matplotlib.pyplot as plt

from src.env.warehouse_env import WarehouseEnv
from src.analytics import RewardTracker

with open('/content/smart-warehouse/configs/env_config.yaml') as f:
    env_config = yaml.safe_load(f)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

env = WarehouseEnv(env_config)
OBS_DIM    = env.obs_dim
ACTION_DIM = env.action_dim
N_AGENTS   = env.n_agents
STATE_DIM  = N_AGENTS * OBS_DIM
MAX_STEPS  = env_config['env'].get('max_steps', 500)
print(f'obs={OBS_DIM}  actions={ACTION_DIM}  agents={N_AGENTS}  state={STATE_DIM}')

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────
HIDDEN_DIM          = 128
N_LAYERS            = 2
MIXING_DIM          = 32
LR                  = 5e-4
GAMMA               = 0.99
MAX_GRAD_NORM       = 10.0
TARGET_UPDATE_INT   = 200
BUFFER_SIZE         = 100_000
BATCH_SIZE          = 128
TRAIN_START         = 1_000
EPSILON_START       = 1.0
EPSILON_END         = 0.05
TOTAL_TIMESTEPS     = 2_000_000
LOG_INTERVAL        = 2_000
EVAL_INTERVAL       = 10_000
EVAL_EPISODES       = 20

In [ ]:
# ── Networks ──────────────────────────────────────────────────────
def build_mlp(in_dim, hidden_dim, out_dim, n_layers):
    gain = nn.init.calculate_gain('relu')
    layers = []
    d = in_dim
    for _ in range(n_layers):
        l = nn.Linear(d, hidden_dim)
        nn.init.orthogonal_(l.weight, gain=gain)
        nn.init.zeros_(l.bias)
        layers += [l, nn.ReLU()]
        d = hidden_dim
    out = nn.Linear(d, out_dim)
    nn.init.orthogonal_(out.weight, gain=1.0)
    nn.init.zeros_(out.bias)
    layers.append(out)
    return nn.Sequential(*layers)

class AgentQNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = build_mlp(OBS_DIM, HIDDEN_DIM, ACTION_DIM, N_LAYERS)
    def forward(self, obs):
        return self.net(obs)

class MixingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hyper_w1 = nn.Sequential(
            nn.Linear(STATE_DIM, MIXING_DIM), nn.ReLU(),
            nn.Linear(MIXING_DIM, N_AGENTS * MIXING_DIM))
        self.hyper_b1 = nn.Linear(STATE_DIM, MIXING_DIM)
        self.hyper_w2 = nn.Sequential(
            nn.Linear(STATE_DIM, MIXING_DIM), nn.ReLU(),
            nn.Linear(MIXING_DIM, MIXING_DIM))
        self.hyper_b2 = nn.Sequential(
            nn.Linear(STATE_DIM, MIXING_DIM), nn.ReLU(),
            nn.Linear(MIXING_DIM, 1))

    def forward(self, agent_qs, state):
        B = agent_qs.size(0)
        w1 = torch.abs(self.hyper_w1(state)).view(B, N_AGENTS, MIXING_DIM)
        b1 = self.hyper_b1(state).unsqueeze(1)
        hidden = F.elu(torch.bmm(agent_qs.unsqueeze(1), w1) + b1)
        w2 = torch.abs(self.hyper_w2(state)).unsqueeze(2)
        b2 = self.hyper_b2(state)
        return torch.bmm(hidden, w2).squeeze(2).squeeze(1) + b2.squeeze(1)

agent_net    = AgentQNet().to(DEVICE)
target_net   = AgentQNet().to(DEVICE)
target_net.load_state_dict(agent_net.state_dict())
mixer        = MixingNet().to(DEVICE)
target_mixer = MixingNet().to(DEVICE)
target_mixer.load_state_dict(mixer.state_dict())

params = list(agent_net.parameters()) + list(mixer.parameters())
optimizer = torch.optim.Adam(params, lr=LR)
print('Networks ready.')

In [ ]:
# ── Replay Buffer & Obs Normalization ─────────────────────────────
class ReplayBuffer:
    def __init__(self, maxlen):
        self.buf = deque(maxlen=maxlen)
    def push(self, obs, actions, rewards, next_obs, dones, state, next_state):
        self.buf.append((obs.astype(np.float32), actions.astype(np.int64),
                         rewards.astype(np.float32), next_obs.astype(np.float32),
                         dones.astype(np.float32), state.astype(np.float32),
                         next_state.astype(np.float32)))
    def sample(self, n):
        batch = random.sample(self.buf, n)
        obs,actions,rewards,next_obs,dones,states,next_states = zip(*batch)
        to_t = lambda x: torch.tensor(np.stack(x), device=DEVICE)
        return (to_t(obs), torch.tensor(np.stack(actions), device=DEVICE),
                to_t(rewards), to_t(next_obs), to_t(dones),
                to_t(states), to_t(next_states))
    def __len__(self): return len(self.buf)

class RunningMeanStd:
    def __init__(self, shape):
        self.mean = np.zeros(shape, np.float64)
        self.var  = np.ones(shape,  np.float64)
        self.count = 1e-4
    def update(self, x):
        bm, bv, bc = x.mean(0), x.var(0), len(x)
        d = bm - self.mean; tot = self.count + bc
        self.mean = self.mean + d * bc / tot
        self.var  = (self.var*self.count + bv*bc + d**2*self.count*bc/tot) / tot
        self.count = tot
    def normalize(self, x):
        return ((x - self.mean) / (np.sqrt(self.var) + 1e-8)).astype(np.float32)

buffer  = ReplayBuffer(BUFFER_SIZE)
obs_rms = RunningMeanStd((OBS_DIM,))
print('Buffer & normalizer ready.')

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────
epsilon = EPSILON_START
ep_rewards = []
ep_rewards_cur = np.zeros(N_AGENTS)
ep_steps = 0
ep_count = 0
update_count = 0
last_loss = 0.0
train_log = []
recent = []

obs = env.reset()
t_start = time.time()

for t in range(1, TOTAL_TIMESTEPS + 1):
    # Select action
    raw = np.stack(obs)
    obs_rms.update(raw)
    norm = obs_rms.normalize(raw)

    if np.random.rand() < epsilon:
        actions = np.random.randint(0, ACTION_DIM, size=N_AGENTS)
    else:
        with torch.no_grad():
            q = agent_net(torch.tensor(norm, device=DEVICE))
        actions = q.argmax(dim=-1).cpu().numpy()

    next_obs, rews, dones, _ = env.step(actions.tolist())

    # Store transition
    raw_next  = np.stack(next_obs)
    norm_next = obs_rms.normalize(raw_next)
    state      = norm.flatten()
    next_state = norm_next.flatten()
    buffer.push(norm, actions, np.array(rews), norm_next,
                np.array(dones, dtype=np.float32), state, next_state)

    ep_rewards_cur += rews
    ep_steps += 1
    obs = next_obs

    # Train
    if len(buffer) >= TRAIN_START:
        o, a, r, no, d, s, ns = buffer.sample(BATCH_SIZE)
        B = o.size(0)

        # Current Q
        q_vals   = agent_net(o.view(B*N_AGENTS, OBS_DIM)).view(B, N_AGENTS, ACTION_DIM)
        chosen_q = q_vals.gather(2, a.unsqueeze(2)).squeeze(2)

        # Target Q (Double DQN)
        with torch.no_grad():
            nq_online = agent_net(no.view(B*N_AGENTS, OBS_DIM)).view(B, N_AGENTS, ACTION_DIM)
            best_a    = nq_online.argmax(dim=2, keepdim=True)
            nq_target = target_net(no.view(B*N_AGENTS, OBS_DIM)).view(B, N_AGENTS, ACTION_DIM)
            nq_chosen = nq_target.gather(2, best_a).squeeze(2)

        q_tot  = mixer(chosen_q, s)
        with torch.no_grad():
            nq_tot = target_mixer(nq_chosen, ns)

        team_r    = r.sum(dim=1)
        team_done = d.max(dim=1).values
        target_q  = team_r + GAMMA * nq_tot * (1 - team_done)

        loss = nn.MSELoss()(q_tot, target_q)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(params, MAX_GRAD_NORM)
        optimizer.step()
        last_loss = loss.item()
        update_count += 1

        if update_count % TARGET_UPDATE_INT == 0:
            target_net.load_state_dict(agent_net.state_dict())
            target_mixer.load_state_dict(mixer.state_dict())

    # Epsilon decay
    epsilon = max(EPSILON_END, EPSILON_START - (EPSILON_START - EPSILON_END) * t / TOTAL_TIMESTEPS)

    # Episode end
    if all(dones):
        team_r = float(ep_rewards_cur.sum())
        ep_rewards.append(team_r)
        recent.append(team_r)
        ep_count += 1
        ep_rewards_cur = np.zeros(N_AGENTS)
        ep_steps = 0
        obs = env.reset()

    # Logging
    if t % LOG_INTERVAL == 0 and recent:
        fps = t / (time.time() - t_start)
        print(f"  t={t:>9,}  ep={ep_count:<5}  mean_r(50)={np.mean(recent[-50:]):>7.3f}  "
              f"loss={last_loss:>8.4f}  eps={epsilon:.3f}  fps={fps:>5.0f}")

    # Evaluation
    if t % EVAL_INTERVAL == 0:
        eval_env = WarehouseEnv(env_config)
        eval_rs = []
        for _ in range(EVAL_EPISODES):
            o2 = eval_env.reset(); er = 0
            for _ in range(MAX_STEPS):
                n2 = obs_rms.normalize(np.stack(o2))
                with torch.no_grad():
                    a2 = agent_net(torch.tensor(n2, device=DEVICE)).argmax(dim=-1).cpu().numpy()
                o2, r2, d2, _ = eval_env.step(a2.tolist())
                er += sum(r2)
                if all(d2): break
            eval_rs.append(er)
        eval_env.close()
        em = np.mean(eval_rs)
        print(f"  [eval] t={t:>9,}  reward={em:.3f} ± {np.std(eval_rs):.3f}  "
              f"max={max(eval_rs):.1f}  pos={np.mean(np.array(eval_rs)>0):.1%}")
        train_log.append({'t': t, 'eval_mean': em, 'eval_std': float(np.std(eval_rs)),
                          'loss': last_loss, 'epsilon': epsilon})

env.close()
print('Training done!')

In [ ]:
# ── Training Curve ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('QMIX Training — Smart Warehouse', fontsize=14, fontweight='bold')

if train_log:
    ts    = [e['t']         for e in train_log]
    means = [e['eval_mean'] for e in train_log]
    stds  = [e['eval_std']  for e in train_log]
    axes[0].plot(ts, means, color='#1976D2', linewidth=1.5, label='QMIX eval mean')
    axes[0].fill_between(ts, np.array(means)-np.array(stds),
                         np.array(means)+np.array(stds), alpha=0.2, color='#1976D2')
    axes[0].axhline(0.443, color='#66BB6A', linestyle='--', label='Greedy (0.443)')
    axes[0].axhline(0.066, color='#EF5350', linestyle='--', label='Random (0.066)')
    axes[0].set_xlabel('Timestep'); axes[0].set_ylabel('Team Total Reward')
    axes[0].set_title('Eval Reward over Training'); axes[0].legend(); axes[0].grid(alpha=0.3)

if ep_rewards:
    axes[1].plot(ep_rewards, color='#90CAF9', alpha=0.3, linewidth=0.5)
    w = min(100, max(1, len(ep_rewards)//10))
    if len(ep_rewards) >= w:
        rolled = np.convolve(ep_rewards, np.ones(w)/w, mode='valid')
        axes[1].plot(range(w-1, len(ep_rewards)), rolled, color='#1976D2',
                     linewidth=1.5, label=f'Rolling mean ({w})')
    axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Team Total Reward')
    axes[1].set_title('Training Episode Rewards'); axes[1].legend(); axes[1].grid(alpha=0.3)

if train_log:
    axes[2].plot(ts, [e['loss']    for e in train_log], color='#EF5350', label='TD loss')
    ax2 = axes[2].twinx()
    ax2.plot(ts, [e['epsilon'] for e in train_log], color='#AB47BC', linestyle='--', label='Epsilon')
    ax2.set_ylabel('Epsilon', color='#AB47BC')
    axes[2].set_xlabel('Timestep'); axes[2].set_ylabel('Loss')
    axes[2].set_title('TD Loss & Epsilon'); axes[2].grid(alpha=0.3)

plt.tight_layout()
os.makedirs('/content/smart-warehouse/results/plots', exist_ok=True)
plt.savefig('/content/smart-warehouse/results/plots/qmix_training_curve.png', dpi=150)
plt.show()
print('Plot saved.')

In [ ]:
# ── Final Evaluation Summary ──────────────────────────────────────
eval_env = WarehouseEnv(env_config)
tracker  = RewardTracker(n_agents=N_AGENTS)

for ep in range(300):
    o = eval_env.reset()
    tracker.start_episode()
    for _ in range(MAX_STEPS):
        n = obs_rms.normalize(np.stack(o))
        with torch.no_grad():
            a = agent_net(torch.tensor(n, device=DEVICE)).argmax(dim=-1).cpu().numpy()
        o, r, d, _ = eval_env.step(a.tolist())
        tracker.record_step(r)
        if all(d): break
    tracker.end_episode()

eval_env.close()
s = tracker.summary()
print('=' * 60)
print('  QMIX Final Evaluation (300 episodes)')
print('=' * 60)
print(f"  Mean reward : {s['team_total_reward']['mean']:.4f}")
print(f"  Std         : {s['team_total_reward']['std']:.4f}")
print(f"  Min / Max   : {s['team_total_reward']['min']:.4f} / {s['team_total_reward']['max']:.4f}")
print(f"  Positive ep : {sum(1 for e in tracker.episodes if e['team_total_reward']>0)}/300")
print('=' * 60)
print(f"  Greedy baseline : 0.4430")
print(f"  Random baseline : 0.0658")
print(f"  QMIX            : {s['team_total_reward']['mean']:.4f}")

In [ ]:
# ── Save results & Download ───────────────────────────────────────
os.makedirs('/content/smart-warehouse/results/logs', exist_ok=True)
tracker.save('/content/smart-warehouse/results/logs/qmix_eval_rewards.json')
tracker.save_csv('/content/smart-warehouse/results/logs/qmix_eval_rewards.csv')

from google.colab import files
files.download('/content/smart-warehouse/results/logs/qmix_eval_rewards.json')
files.download('/content/smart-warehouse/results/plots/qmix_training_curve.png')